# AI6130 Politeness Bias — Full Execution: Gemma-3-27B

## 1. Environment Initialization & Repository Sync

In [1]:
import os
from pathlib import Path
from google.colab import drive, userdata

In [2]:
# Mount Google Drive for real-time persistent storage of raw JSONL outputs
drive.mount("/content/drive")
DRIVE_BACKUP_DIR = Path("/content/drive/MyDrive/AI6130 Group Project/results/raw")
DRIVE_BACKUP_DIR.mkdir(parents=True, exist_ok=True)

# Load tokens securely from Colab Secrets into environment variables
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["GITHUB_PAT"] = userdata.get("GITHUB_PAT")

# Suppress Hugging Face download progress bars to prevent spamming blank lines/carriage returns in Colab cell output
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

Mounted at /content/drive


## 2. Repository Sync & Colab Native Setup

In [3]:
GITHUB_USER = "HJunyuan"
REPO_NAME = "ai6130-group-9-project"
BRANCH_NAME = "main"

In [4]:
%cd /content
if not Path(f"/content/{REPO_NAME}").exists():
    !git clone -b {BRANCH_NAME} https://{os.environ["GITHUB_PAT"]}@github.com/{GITHUB_USER}/{REPO_NAME}.git
%cd /content/{REPO_NAME}

!git pull origin {BRANCH_NAME}

/content
Cloning into 'ai6130-group-9-project'...
remote: Enumerating objects: 332, done.
remote: Counting objects: 100% (101/101), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 332 (delta 70), reused 77 (delta 60), pack-reused 231 (from 1)
Receiving objects: 100% (332/332), 1.20 MiB | 21.62 MiB/s, done.
Resolving deltas: 100% (175/175), done.
/content/ai6130-group-9-project
From https://github.com/HJunyuan/ai6130-group-9-project
 * branch            main       -> FETCH_HEAD
Already up to date.


In [5]:
%%capture
!pip install -U bottleneck  # to silence Pandas warning
!pip install -q uv
!uv pip install --system -e .

## 3. Pipeline Architecture: Fault Tolerance & Cache Management

In [6]:
import shutil
from datetime import datetime, timezone, timedelta
from pathlib import Path

In [7]:
SGT = timezone(timedelta(hours=8))

In [8]:
def get_drive_run_file(safe_model_name: str, dataset_tag: str = "main") -> Path:
    """
    Checks Google Drive for an existing JSONL file for this model + dataset_tag.
    If found, returns that path so generate.py automatically resumes from the last saved batch.
    Otherwise, creates a new SGT-timestamped filename in Google Drive.
    """
    prefix = f"{safe_model_name}__" if dataset_tag == "main" else f"{safe_model_name}__yesman__"
    existing_drive_files = sorted(DRIVE_BACKUP_DIR.glob(f"{prefix}*.jsonl"))

    if dataset_tag == "main":
        existing_drive_files = [
            f for f in existing_drive_files
            if "__yesman__" not in f.name and not f.name.startswith("pilot_")
        ]

    if existing_drive_files:
        latest_drive_file = existing_drive_files[-1]
        print(f"[RESUME] Found existing file on Google Drive: {latest_drive_file.name}")
        return latest_drive_file

    run_id = datetime.now(SGT).strftime("%Y%m%d-%H%M%S")
    return DRIVE_BACKUP_DIR / f"{prefix}{run_id}.jsonl"


def run_model_pipeline(
    model_id: str,
    batch_size: int = 8,
    run_yesman: bool = True,
    purge_cache_after: bool = True,
):
    safe_name = model_id.replace("/", "--")
    local_raw_dir = Path("results/raw")
    local_raw_dir.mkdir(parents=True, exist_ok=True)

    # Main Dataset (1,150 prompts) — writes directly to Google Drive per batch
    main_drive_out = get_drive_run_file(safe_name, dataset_tag="main")
    print(f"\n============================================================")
    print(f"Running Main Dataset (1,150 prompts) on {model_id}")
    print(f"Drive Output File: {main_drive_out}")
    print(f"============================================================")
    %run src/ai6130_group_9_project/run/generate.py --model "{model_id}" --input data/prompts_v1.jsonl --batch-size {batch_size} --output "{main_drive_out}"
    shutil.copy2(main_drive_out, local_raw_dir / main_drive_out.name)
    print(f"[SAVED] Completed {main_drive_out.name} on Google Drive.")

    # Yes-Man Sycophancy Stretch Dataset (300 prompts) — writes directly to Google Drive per batch
    if run_yesman:
        yesman_drive_out = get_drive_run_file(safe_name, dataset_tag="yesman")
        print(f"\n============================================================")
        print(f"Running Yes-Man Dataset (300 prompts) on {model_id}")
        print(f"Drive Output File: {yesman_drive_out}")
        print(f"============================================================")
        %run src/ai6130_group_9_project/run/generate.py --model "{model_id}" --input data/prompts_yesman_v1.jsonl --batch-size {batch_size} --output "{yesman_drive_out}"
        shutil.copy2(yesman_drive_out, local_raw_dir / yesman_drive_out.name)
        print(f"[SAVED] Completed {yesman_drive_out.name} on Google Drive.")

    # Purge HF model cache
    if purge_cache_after:
        print(f"\n[CLEANUP] Purging Hugging Face weight cache for {model_id}...")
        !rm -rf ~/.cache/huggingface/hub/models--*
        !df -h /content

## 4. Execution Phase: Gemma-3-27B-IT

In [9]:
run_model_pipeline(
    model_id="google/gemma-3-27b-it",
    batch_size=4,
    run_yesman=True,
    purge_cache_after=True,
)


Running Main Dataset (1,150 prompts) on google/gemma-3-27b-it
Drive Output File: /content/drive/MyDrive/AI6130 Group Project/results/raw/google--gemma-3-27b-it__20261028-105010.jsonl
[INIT] Model: google/gemma-3-27b-it | Device: cuda | Dtype: torch.bfloat16 | Batch Size: 4 | Run ID: 20261028-105010


Generating (google--gemma-3-27b-it):   0%|          | 0/288 [00:00<?, ?it/s]


[COMPLETE] Saved outputs to /content/drive/MyDrive/AI6130 Group Project/results/raw/google--gemma-3-27b-it__20261028-105010.jsonl
[REPORT LOG] Date: 28 Oct | Run ID: 20261028-105010 | Model: google/gemma-3-27b-it | Revision: 005ad3404e | Items: 1150
[SAVED] Completed google--gemma-3-27b-it__20261028-105010.jsonl on Google Drive.

Running Yes-Man Dataset (300 prompts) on google/gemma-3-27b-it
Drive Output File: /content/drive/MyDrive/AI6130 Group Project/results/raw/google--gemma-3-27b-it__yesman__20261028-134523.jsonl
[INIT] Model: google/gemma-3-27b-it | Device: cuda | Dtype: torch.bfloat16 | Batch Size: 4 | Run ID: 20261028-134523


Generating (google--gemma-3-27b-it):   0%|          | 0/75 [00:00<?, ?it/s]


[COMPLETE] Saved outputs to /content/drive/MyDrive/AI6130 Group Project/results/raw/google--gemma-3-27b-it__yesman__20261028-134523.jsonl
[REPORT LOG] Date: 28 Oct | Run ID: 20261028-134523 | Model: google/gemma-3-27b-it | Revision: 005ad3404e | Items: 300
[SAVED] Completed google--gemma-3-27b-it__yesman__20261028-134523.jsonl on Google Drive.

[CLEANUP] Purging Hugging Face weight cache for google/gemma-3-27b-it...
Filesystem      Size  Used Avail Use% Mounted on
overlay         236G   95G  142G  40% /
